# Reparations Initiatives Pipeline

This notebook ingests, enriches, and publishes reparations initiative data to ArcGIS Online.

**Run order:** Execute all cells top-to-bottom. Each step cell can also be re-run independently
(assuming Drive is mounted and secrets are loaded).

**Prerequisites:**
- Google Drive mounted (Cell 2)
- Four Colab Secrets configured (Cell 3): `CONGRESS_API_KEY`, `AGOL_USERNAME`, `AGOL_PASSWORD`, `GOOGLE_CIVIC_API_KEY`

See `README.md` in the repo for full documentation.

---
## Cell 1 — Install Dependencies

Installs all required packages into the Colab runtime. Re-run if you encounter import errors.

**Expected duration:** 1–3 minutes on first run; faster on subsequent runs if packages are cached.

In [ ]:
!apt-get install -q -y chromium-chromedriver
!pip install -q geopandas shapely playwright nest_asyncio selenium webdriver-manager pandas requests beautifulsoup4
!playwright install chromium

---
## Cell 2 — Mount Google Drive & Set Data Base Path

Mounts your Google Drive and sets `REPARATIONS_DATA_BASE` so all modules write
persistent data to Drive (not to Colab's ephemeral `/content/` filesystem).

**All pipeline outputs are written to:**
`My Drive / reparations_gis / data /`

**First-run note:** Google will prompt you to authorize Drive access.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["REPARATIONS_DATA_BASE"] = "/content/drive/MyDrive/reparations_gis/data"
os.makedirs(os.environ["REPARATIONS_DATA_BASE"], exist_ok=True)
print(f"Data base path: {os.environ['REPARATIONS_DATA_BASE']}")

---
## Cell 3 — Load Secrets

Reads all required credentials from Colab Secrets (the key icon in the left sidebar).

**Required secrets:**
- `CONGRESS_API_KEY` — Congress.gov API key (free at [api.congress.gov](https://api.congress.gov))
- `AGOL_USERNAME` — Your ArcGIS Online username
- `AGOL_PASSWORD` — Your ArcGIS Online password
- `GOOGLE_CIVIC_API_KEY` — Google Civic Information API key (fallback only; free at [console.cloud.google.com](https://console.cloud.google.com))

**If any secret is missing**, this cell raises a `ValueError` with setup instructions.

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = [
    "CONGRESS_API_KEY",
    "AGOL_USERNAME",
    "AGOL_PASSWORD",
    "GOOGLE_CIVIC_API_KEY",
]

missing = []
for secret in REQUIRED_SECRETS:
    try:
        val = userdata.get(secret)
        if not val:
            missing.append(secret)
        else:
            os.environ[secret] = val
    except Exception:
        missing.append(secret)

if missing:
    raise ValueError(
        f"Missing Colab Secrets: {missing}\n"
        "To add secrets:\n"
        "  1. Click the key icon (🔑) in the left sidebar\n"
        "  2. Click 'Add new secret'\n"
        "  3. Enter the secret name and value\n"
        "  4. Enable access for this notebook\n"
        "  5. Re-run this cell"
    )

print("All secrets loaded successfully.")

---
## Cell 4 — Clone Repo & Import Modules

Clones the pipeline repository and imports the three step modules.

**Edit the `REPO_URL` variable** if you have forked the repo to your own GitHub account.

In [ ]:
import subprocess
import sys
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s %(message)s",
    handlers=[logging.StreamHandler()],
)

REPO_URL = "https://github.com/land-rover/reparations-tracker"
REPO_DIR = "/content/reparations-tracker"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Remove stale bytecode
subprocess.run(["find", REPO_DIR, "-type", "d", "-name", "__pycache__",
                "-exec", "rm", "-rf", "{}", "+"], capture_output=True)

# Evict any already-imported pipeline modules so fresh source is used
for _mod in list(sys.modules.keys()):
    if _mod.startswith(("ingest", "enrich", "publish", "config")):
        del sys.modules[_mod]

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ingest.fetch_table import run as fetch
from enrich.enrich_initiatives import run as enrich
from publish.publish_to_agol import run as publish
import config.settings as settings

print("Modules imported successfully.")
print(f"Data base: {settings.DATA_BASE}")

---
## Step 1 — Fetch Initiatives Table

**What it does:** Fetches the reparations initiatives table from
[reparationsresources.com/table](https://www.reparationsresources.com/table).

**Strategies tried in order:**
1. ArcGIS REST API (direct JSON — fastest and most reliable)
2. Playwright headless Chromium (full JS rendering)
3. requests + BeautifulSoup (plain HTML)

**Output:** `{REPARATIONS_DATA_BASE}/initiatives_raw.json`

**Expected duration:** 10–60 seconds depending on site response time.

**Note:** If the source site has changed its structure, check the logs for which
strategy succeeded and whether all expected fields were captured.

In [ ]:
records = fetch(settings)
print(f"Fetched {len(records)} initiative records.")
if records:
    print("Sample record fields:", list(records[0].keys()))

---
## Step 2 — Enrich Initiatives

**What it does:** For each initiative, resolves:
- City boundary polygon (TIGER/Line Place shapefile)
- Intersection with 119th Congress district polygons → one point per overlapping district
- Current U.S. House Representative (Congress.gov API)
- Current U.S. Senators (Congress.gov API)
- H.R. 40 co-sponsorship status for each legislator

**Inputs:**
- `initiatives_raw.json` (from Step 1)
- TIGER/Line shapefiles (downloaded to Drive cache on first run)

**Outputs:**
- `initiatives_enriched.geojson` — one GeoJSON Point per initiative×district intersection
- `legislators_summary.json` — one record per legislator with `initiative_count` and `hr40_position`
- `geocode_cache.json` — city boundary cache (persists across sessions)

**First-run warning:** This step downloads approximately **600 MB** of TIGER/Line Place
shapefiles (one zip per state) to your Google Drive. This can take 10–20 minutes on
a slow connection. Subsequent runs load directly from Drive cache and are fast.

**Expected duration:** 15–30 minutes (first run) / 3–10 minutes (subsequent runs)

In [ ]:
enrich(settings)
print("Enrichment complete. Check logs above for any warnings or fallbacks.")

---
## Step 3 — Publish to ArcGIS Online

**What it does:** Creates or updates four ArcGIS Online layers using the REST API:

| Layer | Type | Description |
|-------|------|-------------|
| `Reparations_Initiatives_Points` | Point Feature Layer | One point per initiative×district intersection |
| `Reparations_Legislators_Summary` | Hosted Table | One row per legislator |
| `Reparations_Congressional_Districts` | Polygon Feature Layer | 119th Congress districts with initiative counts |
| `Reparations_Senate_States` | Polygon Feature Layer | States with senator info and delegation alignment |

**First run:** Creates new items and writes their IDs back to `config/settings.py`.
**Subsequent runs:** Overwrites existing layer data in place (preserves item IDs,
so any maps or dashboards referencing these layers continue to work).

**Safety check:** If the new record count is less than 80% of the existing count,
the publish is aborted with an error to prevent a bad scrape from overwriting good data.

**Inputs:**
- `initiatives_enriched.geojson` and `legislators_summary.json` (from Step 2)
- TIGER/Line district and state boundaries (already cached from Step 2)
- `AGOL_USERNAME` and `AGOL_PASSWORD` environment variables

**Expected duration:** 5–15 minutes

In [ ]:
publish(settings)
print("Publish complete. Check logs above for item IDs and any errors.")